# 🔬 SPECTRA — pix2pix GAN Training Notebook

**Bharatiya Antariksh Hackathon 2026 | PS 10: Infrared Image Colorization**

This notebook trains a pix2pix GAN to colorize infrared images.

## Instructions:
1. Upload your preprocessed dataset to Google Drive under `SPECTRA/data/`
2. Set GPU runtime: Runtime → Change runtime type → T4 GPU
3. Run all cells (takes ~4-6 hours for 200 epochs)
4. Download `generator_best.pth` from Drive when done

In [ ]:
# ─── 1. Check GPU ─────────────────────────────────────
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️ No GPU detected! Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ─── 2. Mount Google Drive & Setup Fast Storage ───────────
from google.colab import drive
import os

drive.mount('/content/drive')

# Set paths
DRIVE_ROOT = '/content/drive/MyDrive/SPECTRA'
DRIVE_DATA = f'{DRIVE_ROOT}/data'
CHECKPOINT_DIR = f'{DRIVE_ROOT}/checkpoints'
SAMPLE_DIR = f'{DRIVE_ROOT}/samples'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(SAMPLE_DIR, exist_ok=True)

# Verify dataset exists on Drive
if not os.path.exists(DRIVE_DATA):
    print('❌ Dataset not found on Google Drive!')
    print(f'   Expected: {DRIVE_DATA}')
    print('   Upload your preprocessed dataset to Google Drive under SPECTRA/data/')
else:
    print(f'✅ Dataset found on Drive!')
    
    LOCAL_DATA = '/content/data_local'
    # Check if we already copied it completely
    needs_copy = True
    if os.path.exists(f'{LOCAL_DATA}/val/B'):
        if len(os.listdir(f'{LOCAL_DATA}/train/A')) > 1000:
            needs_copy = False
            
    if needs_copy:
        print('⏳ Copying 8,000+ images from Drive to Colab Ultra-Fast SSD...')
        print('   (This will take ~5 minutes, but it will save you over 20 hours of training time!)')
        !rm -rf {LOCAL_DATA}  # clear any partial copies
        !cp -r {DRIVE_DATA} {LOCAL_DATA}
        print('✅ Copy complete!')
    else:
        print('✅ Local dataset already exists and is ready!')
        
    # Point the training script to the fast local data!
    DATA_ROOT = LOCAL_DATA


In [ ]:
# ─── 3. Install dependencies ─────────────────────────
!pip install -q scikit-image torchvision pillow tqdm

In [ ]:
# ─── 4. Define Model Architecture ────────────────────
import torch
import torch.nn as nn


class UNetDownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, use_norm=True):
        super().__init__()
        layers = [nn.Conv2d(in_ch, out_ch, 4, 2, 1, bias=False)]
        if use_norm:
            layers.append(nn.BatchNorm2d(out_ch))
        layers.append(nn.LeakyReLU(0.2, True))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class UNetUpBlock(nn.Module):
    def __init__(self, in_ch, out_ch, use_dropout=False):
        super().__init__()
        layers = [
            nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1, bias=False),
            nn.BatchNorm2d(out_ch),
        ]
        if use_dropout:
            layers.append(nn.Dropout(0.5))
        layers.append(nn.ReLU(True))
        self.block = nn.Sequential(*layers)

    def forward(self, x, skip):
        x = self.block(x)
        return torch.cat([x, skip], 1)


class UNetGenerator(nn.Module):
    def __init__(self, in_ch=1, out_ch=3, f=64):
        super().__init__()
        self.e1 = UNetDownBlock(in_ch, f, False)
        self.e2 = UNetDownBlock(f, f*2)
        self.e3 = UNetDownBlock(f*2, f*4)
        self.e4 = UNetDownBlock(f*4, f*8)
        self.e5 = UNetDownBlock(f*8, f*8)
        self.e6 = UNetDownBlock(f*8, f*8)
        self.e7 = UNetDownBlock(f*8, f*8)
        self.bottleneck = nn.Sequential(nn.Conv2d(f*8, f*8, 4, 2, 1, bias=False), nn.ReLU(True))
        self.d7 = UNetUpBlock(f*8, f*8, True)
        self.d6 = UNetUpBlock(f*16, f*8, True)
        self.d5 = UNetUpBlock(f*16, f*8, True)
        self.d4 = UNetUpBlock(f*16, f*8)
        self.d3 = UNetUpBlock(f*16, f*4)
        self.d2 = UNetUpBlock(f*8, f*2)
        self.d1 = UNetUpBlock(f*4, f)
        self.output = nn.Sequential(nn.ConvTranspose2d(f*2, out_ch, 4, 2, 1), nn.Tanh())

    def forward(self, x):
        e1=self.e1(x); e2=self.e2(e1); e3=self.e3(e2); e4=self.e4(e3)
        e5=self.e5(e4); e6=self.e6(e5); e7=self.e7(e6)
        b = self.bottleneck(e7)
        d7=self.d7(b,e7); d6=self.d6(d7,e6); d5=self.d5(d6,e5); d4=self.d4(d5,e4)
        d3=self.d3(d4,e3); d2=self.d2(d3,e2); d1=self.d1(d2,e1)
        return self.output(d1)


class PatchGANDiscriminator(nn.Module):
    def __init__(self, in_ch=4, f=64):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(in_ch, f, 4, 2, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(f, f*2, 4, 2, 1, bias=False), nn.BatchNorm2d(f*2), nn.LeakyReLU(0.2, True),
            nn.Conv2d(f*2, f*4, 4, 2, 1, bias=False), nn.BatchNorm2d(f*4), nn.LeakyReLU(0.2, True),
            nn.Conv2d(f*4, f*8, 4, 1, 1, bias=False), nn.BatchNorm2d(f*8), nn.LeakyReLU(0.2, True),
            nn.Conv2d(f*8, 1, 4, 1, 1),
        )

    def forward(self, ir, rgb):
        return self.model(torch.cat([ir, rgb], 1))


def init_weights(model, std=0.02):
    for m in model.modules():
        name = m.__class__.__name__
        if 'Conv' in name:
            nn.init.normal_(m.weight.data, 0.0, std)
        elif 'BatchNorm' in name:
            nn.init.normal_(m.weight.data, 1.0, std)
            nn.init.constant_(m.bias.data, 0)


# Test
G = UNetGenerator()
D = PatchGANDiscriminator()
x = torch.randn(1, 1, 256, 256)
print(f'Generator output: {G(x).shape}')
print(f'Generator params: {sum(p.numel() for p in G.parameters()):,}')
print(f'Discriminator params: {sum(p.numel() for p in D.parameters()):,}')

In [ ]:
# ─── 5. Dataset ──────────────────────────────────────
import os
import random
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
import torchvision.transforms as T


class IRColorDataset(Dataset):
    def __init__(self, root, split='train', size=256, jitter=286):
        self.split = split
        self.size = size
        self.jitter = jitter
        self.dir_ir = Path(root) / split / 'A'
        self.dir_rgb = Path(root) / split / 'B'
        ir_files = set(os.listdir(self.dir_ir))
        rgb_files = set(os.listdir(self.dir_rgb))
        self.filenames = sorted(ir_files & rgb_files)
        print(f'[{split.upper()}] {len(self.filenames)} paired images')

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        f = self.filenames[idx]
        ir = Image.open(self.dir_ir / f).convert('L')
        rgb = Image.open(self.dir_rgb / f).convert('RGB')

        if self.split == 'train':
            ir = TF.resize(ir, [self.jitter, self.jitter], T.InterpolationMode.BICUBIC)
            rgb = TF.resize(rgb, [self.jitter, self.jitter], T.InterpolationMode.BICUBIC)
            i, j, h, w = T.RandomCrop.get_params(ir, (self.size, self.size))
            ir = TF.crop(ir, i, j, h, w)
            rgb = TF.crop(rgb, i, j, h, w)
            if random.random() > 0.5:
                ir = TF.hflip(ir)
                rgb = TF.hflip(rgb)
        else:
            ir = TF.resize(ir, [self.size, self.size], T.InterpolationMode.BICUBIC)
            rgb = TF.resize(rgb, [self.size, self.size], T.InterpolationMode.BICUBIC)

        ir_t = TF.to_tensor(ir)
        rgb_t = TF.to_tensor(rgb)
        if ir_t.shape[0] == 3:
            ir_t = ir_t.mean(0, keepdim=True)
        return ir_t * 2 - 1, rgb_t * 2 - 1


train_ds = IRColorDataset(DATA_ROOT, 'train')
val_ds = IRColorDataset(DATA_ROOT, 'val')
train_dl = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=2, pin_memory=True)
val_dl = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=1)

# Show sample
import matplotlib.pyplot as plt
ir, rgb = train_ds[0]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
ax1.imshow((ir.squeeze() + 1) / 2, cmap='gray'); ax1.set_title('IR Input'); ax1.axis('off')
ax2.imshow(((rgb + 1) / 2).permute(1, 2, 0)); ax2.set_title('RGB Target'); ax2.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# ─── 6. Training Setup ────────────────────────────────
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from torchvision.utils import save_image
import numpy as np
import time

device = torch.device('cuda')

# Models
G = UNetGenerator(1, 3, 64).to(device)
D = PatchGANDiscriminator(4, 64).to(device)
init_weights(G)
init_weights(D)

# Optimizers
opt_G = torch.optim.Adam(G.parameters(), lr=0.0002, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=0.0002, betas=(0.5, 0.999))

# Losses
gan_loss = nn.MSELoss()  # LSGAN
l1_loss = nn.L1Loss()
LAMBDA_L1 = 100

print(f'✅ Ready to train on {device} ({torch.cuda.get_device_name(0)})')

In [ ]:
# ─── 7. TRAIN! ──────────────────────────────────────
EPOCHS = 50
SAVE_EVERY = 10
best_psnr = 0.0

print(f'🚀 Training for {EPOCHS} epochs on {len(train_ds)} images')
print('='*60)

for epoch in range(EPOCHS):
    G.train(); D.train()
    g_total, d_total, l1_total = 0, 0, 0
    t0 = time.time()

    for ir, rgb in train_dl:
        ir, rgb = ir.to(device), rgb.to(device)

        # --- Discriminator ---
        opt_D.zero_grad()
        fake = G(ir)
        d_real = D(ir, rgb)
        d_fake = D(ir, fake.detach())
        loss_d = 0.5 * (gan_loss(d_real, torch.ones_like(d_real)) + gan_loss(d_fake, torch.zeros_like(d_fake)))
        loss_d.backward()
        opt_D.step()

        # --- Generator ---
        opt_G.zero_grad()
        d_fake_g = D(ir, fake)
        loss_gan = gan_loss(d_fake_g, torch.ones_like(d_fake_g))
        loss_l1 = l1_loss(fake, rgb)
        loss_g = loss_gan + LAMBDA_L1 * loss_l1
        loss_g.backward()
        opt_G.step()

        g_total += loss_g.item()
        d_total += loss_d.item()
        l1_total += loss_l1.item()

    n = len(train_dl)
    elapsed = time.time() - t0
    print(f'Epoch [{epoch+1:3d}/{EPOCHS}] G: {g_total/n:.4f} D: {d_total/n:.4f} L1: {l1_total/n:.4f} | {elapsed:.0f}s')

    # Save samples
    G.eval()
    with torch.inference_mode():
        v_ir, v_rgb = next(iter(val_dl))
        v_ir, v_rgb = v_ir.to(device), v_rgb.to(device)
        v_fake = G(v_ir)
        comp = torch.cat([v_ir.repeat(1,3,1,1), v_fake, v_rgb], 3)
        save_image((comp+1)/2, f'{SAMPLE_DIR}/epoch_{epoch+1:04d}.png')

    # Validation every 10 epochs
    if (epoch+1) % 10 == 0 or epoch == EPOCHS-1:
        psnrs, ssims = [], []
        with torch.inference_mode():
            for v_ir, v_rgb in val_dl:
                v_fake = G(v_ir.to(device))
                f_np = ((v_fake.squeeze().cpu()+1)/2).clamp(0,1).permute(1,2,0).numpy()
                r_np = ((v_rgb.squeeze()+1)/2).clamp(0,1).permute(1,2,0).numpy()
                psnrs.append(peak_signal_noise_ratio(r_np, f_np, data_range=1.0))
                ssims.append(structural_similarity(r_np, f_np, data_range=1.0, channel_axis=2))
        avg_p, avg_s = np.mean(psnrs), np.mean(ssims)
        print(f'  📊 Val PSNR: {avg_p:.2f} dB | SSIM: {avg_s:.4f}')
        if avg_p > best_psnr:
            best_psnr = avg_p
            torch.save(G.state_dict(), f'{CHECKPOINT_DIR}/generator_best.pth')
            print(f'  🏆 New best! Saved generator_best.pth')

    # Checkpoint
    if (epoch+1) % SAVE_EVERY == 0:
        torch.save({
            'epoch': epoch+1, 'G': G.state_dict(), 'D': D.state_dict(),
            'opt_G': opt_G.state_dict(), 'opt_D': opt_D.state_dict(),
        }, f'{CHECKPOINT_DIR}/epoch_{epoch+1:04d}.pth')
        print(f'  💾 Checkpoint saved')

# Final save
torch.save(G.state_dict(), f'{CHECKPOINT_DIR}/generator_final.pth')
print(f'\n✅ Done! Best PSNR: {best_psnr:.2f} dB')
print(f'   Best: {CHECKPOINT_DIR}/generator_best.pth')
print(f'   Final: {CHECKPOINT_DIR}/generator_final.pth')

In [ ]:
# ─── 8. Visualize Results ────────────────────────────
import matplotlib.pyplot as plt

G.eval()
fig, axes = plt.subplots(4, 3, figsize=(15, 20))
cols = ['IR Input', 'Generated RGB', 'Ground Truth']
for j, c in enumerate(cols):
    axes[0, j].set_title(c, fontsize=14, fontweight='bold')

for i, (ir, rgb) in enumerate(val_dl):
    if i >= 4: break
    with torch.inference_mode():
        fake = G(ir.to(device))
    
    ir_np = ((ir.squeeze()+1)/2).numpy()
    fake_np = ((fake.squeeze().cpu()+1)/2).clamp(0,1).permute(1,2,0).numpy()
    real_np = ((rgb.squeeze()+1)/2).clamp(0,1).permute(1,2,0).numpy()
    
    axes[i, 0].imshow(ir_np, cmap='gray'); axes[i, 0].axis('off')
    axes[i, 1].imshow(fake_np); axes[i, 1].axis('off')
    axes[i, 2].imshow(real_np); axes[i, 2].axis('off')

plt.tight_layout()
plt.savefig(f'{SAMPLE_DIR}/final_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Comparison saved to samples/final_comparison.png')

In [ ]:
# ─── 9. Export to ONNX (for browser inference) ───────
G_export = UNetGenerator(1, 3, 64)
G_export.load_state_dict(torch.load(f'{CHECKPOINT_DIR}/generator_best.pth', map_location='cpu'))
G_export.eval()

dummy = torch.randn(1, 1, 256, 256)
onnx_path = f'{CHECKPOINT_DIR}/spectra_generator.onnx'

torch.onnx.export(
    G_export, dummy, onnx_path,
    export_params=True, opset_version=17,
    do_constant_folding=True,
    input_names=['ir_image'], output_names=['rgb_image'],
    dynamic_axes={'ir_image': {0: 'batch'}, 'rgb_image': {0: 'batch'}},
)

size_mb = os.path.getsize(onnx_path) / (1024*1024)
print(f'✅ ONNX exported: {onnx_path} ({size_mb:.1f} MB)')

# Optional: Quantize
try:
    !pip install -q onnxruntime
    from onnxruntime.quantization import quantize_dynamic, QuantType
    q_path = onnx_path.replace('.onnx', '_quantized.onnx')
    quantize_dynamic(onnx_path, q_path, weight_type=QuantType.QUInt8)
    q_size = os.path.getsize(q_path) / (1024*1024)
    print(f'✅ Quantized: {q_path} ({q_size:.1f} MB, {(1-q_size/size_mb)*100:.0f}% smaller)')
except Exception as e:
    print(f'⚠️ Quantization skipped: {e}')

In [ ]:
# ─── 10. Download Files ───────────────────────────────
print('📥 Download these files from Google Drive:')
print(f'   1. {CHECKPOINT_DIR}/generator_best.pth → backend/model/')
print(f'   2. {CHECKPOINT_DIR}/spectra_generator.onnx → frontend/public/model/')
print(f'   3. {SAMPLE_DIR}/final_comparison.png → for your presentation')
print()
print('🎉 Training complete! Your model is ready for SPECTRA.')